# 01. The EDA Decision Framework

How to systematically decide **WHAT to check**, **WHY to check it**, and **WHAT action to take** when encountering any new dataset.


## 1. Objective
Learn how to approach an unfamiliar dataset using a **decision-first framework** rather than mechanical plotting. By the end of this notebook, you will be able to:
1. Automatically classify columns into an actionable **Feature Taxonomy**.
2. Formulate explicit **EDA Hypotheses** before writing code.
3. Map statistical findings directly into concrete **Feature Engineering & ML Pipeline actions**.


## 2. Dataset & Decision Context
- **Dataset**: Credit Risk & Loan Default (`loan_default.csv`)
- **Row Grain**: A single loan applicant evaluated for credit approval.
- **ML Objective**: Binary Classification to predict `default` (0 = Paid in Full, 1 = Default).
- **Decision Stakes**: High false positives reject profitable customers; high false negatives approve bad credit causing direct financial loss.


## 3. What Should I Check?
Before writing any code, we establish our diagnostic checklist:

| Check | Why It Matters | Downstream Impact |
|---|---|---|
| **Feature Taxonomy** | Dictates valid mathematical operations and chart types | Prevents applying continuous statistics to discrete IDs |
| **Missingness & Sparsity** | Distinguishes random gaps from informative signals | Decides between drop, median imputation, or indicator flags |
| **Distribution Skewness** | Heavy right tails distort linear/logistic regression gradients | Guides $\log(x+1)$ or power transformations |
| **Categorical Cardinality** | High cardinality causes dimensionality explosion in OHE | Guides choice between One-Hot, Target, or Frequency Encoding |
| **Target Class Balance** | Severe imbalance renders standard accuracy misleading | Dictates metric choice (PR-AUC/ROC-AUC) and sampling strategy |


## 4. Technique Breakdown

```
WHAT: Systematic 5-Step Exploratory Data Audit
WHY: Prevents premature feature engineering and catches silent data issues early
WHEN: Always at the start of any new machine learning engagement
WHEN NOT: Never skip this step
HOW: Automated classification -> Univariate scan -> Bivariate relationship -> Decision log
WHAT TO LOOK FOR: Skew > 1.5, missingness > 3%, cardinality > 20, class imbalance < 20%
WHAT ACTION: Log transforms for linear models, indicator flags for missingness, target encoding for high cardinality
```


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

# Load Credit Risk Dataset
df = pd.read_csv('../datasets/credit_risk/loan_default.csv')
print(f"Dataset Shape: {df.shape[0]:,} rows | {df.shape[1]} columns")
df.head()


## 5. Automated Feature Taxonomy Classification
We classify each feature into its structural role to dictate subsequent EDA checks.


In [ ]:
def classify_feature_taxonomy(df, target_col=None, id_cols=None):
    id_cols = id_cols or []
    taxonomy = {}
    
    for col in df.columns:
        if col == target_col:
            taxonomy[col] = 'TARGET'
        elif col in id_cols or 'id' in col.lower():
            taxonomy[col] = 'IDENTIFIER'
        elif pd.api.types.is_datetime64_any_dtype(df[col]) or 'date' in col.lower() or 'time' in col.lower():
            taxonomy[col] = 'DATETIME'
        elif pd.api.types.is_numeric_dtype(df[col]):
            unique_count = df[col].nunique()
            if unique_count <= 5 and set(df[col].dropna().unique()).issubset({0, 1}):
                taxonomy[col] = 'BINARY'
            elif unique_count < 15:
                taxonomy[col] = 'NUMERICAL_DISCRETE'
            else:
                taxonomy[col] = 'NUMERICAL_CONTINUOUS'
        else:
            unique_count = df[col].nunique()
            if unique_count <= 10:
                taxonomy[col] = 'CATEGORICAL_LOW_CARD'
            else:
                taxonomy[col] = 'CATEGORICAL_HIGH_CARD'
                
    summary = pd.DataFrame({
        'Feature': list(taxonomy.keys()),
        'Taxonomy': list(taxonomy.values()),
        'Dtype': [df[col].dtype for col in taxonomy.keys()],
        'Missing_Count': [df[col].isna().sum() for col in taxonomy.keys()],
        'Missing_Pct': [round(df[col].isna().mean() * 100, 2) for col in taxonomy.keys()],
        'Unique_Values': [df[col].nunique() for col in taxonomy.keys()],
        'Skewness': [round(df[col].skew(), 2) if taxonomy[col] in ['NUMERICAL_CONTINUOUS', 'NUMERICAL_DISCRETE'] else np.nan for col in taxonomy.keys()]
    })
    return summary

tax_df = classify_feature_taxonomy(df, target_col='default', id_cols=['applicant_id'])
tax_df


## 6. Visualizing the Diagnostic Audit


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Taxonomy Breakdown
tax_counts = tax_df['Taxonomy'].value_counts()
axes[0].barh(tax_counts.index, tax_counts.values, color='#2b5c8f')
axes[0].set_title('Feature Taxonomy Distribution')
axes[0].set_xlabel('Number of Features')

# 2. Skewness across Continuous Features
num_cols = tax_df[tax_df['Taxonomy'] == 'NUMERICAL_CONTINUOUS']['Feature']
skews = df[num_cols].skew().sort_values(ascending=False)
colors = ['#d95f02' if s > 1.0 else '#2b5c8f' for s in skews]
axes[1].barh(skews.index, skews.values, color=colors)
axes[1].axvline(1.0, color='red', linestyle='--', label='Skew Threshold (|s| > 1.0)')
axes[1].set_title('Numerical Feature Skewness')
axes[1].set_xlabel('Skewness Coefficient')
axes[1].legend()

plt.tight_layout()
plt.show()


## 7. Interpretation & Decision Log

### What did we find?
1. **Target Balance**: `default` has an event rate of ~14.0% (moderately imbalanced). Standard accuracy is not viable; use ROC-AUC and PR-AUC.
2. **Heavy Skewness**: `income` (skew = 2.1) and `existing_debt` (skew = 1.8) exhibit heavy right tails.
3. **Missing Values**: `employment_length_years` has 7% missingness; `existing_debt` has 3% missingness.
4. **Categorical Features**: `home_ownership` (4 levels) and `loan_purpose` (7 levels) are low cardinality.

### Explicit Decision
> [!IMPORTANT]
> **Decision Rule**:
> - **Because** `income` and `existing_debt` are heavily right-skewed, we **will** evaluate $\log(x+1)$ transformations for linear/distance-based models, and create a normalized Debt-to-Income (DTI) ratio feature.
> - **Because** `employment_length_years` missingness is non-trivial (7%), we **will** create an explicit binary indicator `employment_length_isna` prior to median imputation.
> - **Because** `loan_purpose` has low cardinality (7 levels), we **will** use One-Hot Encoding with `drop_first=True` for linear models.


## 8. Decision Table: General Framework

| Feature Taxonomy | Primary EDA Checks | Common Red Flags | Recommended Action |
|---|---|---|---|
| **Numerical Continuous** | Histogram, KDE, Boxplot, Skewness, Outliers | Skew $> 1.5$, Extreme IQR outliers | $\log(x+1)$ transform, RobustScaler, domain capping |
| **Numerical Discrete** | Value counts, zero-inflation, cardinality | High zero proportion (> 40%) | Create binary flag `is_zero` + continuous transform |
| **Categorical Low Card** | Frequency bar plot, target rate per group | Rare categories (< 1% freq) | Group rare levels into "Other" $\rightarrow$ One-Hot Encode |
| **Categorical High Card** | Cardinality count, target separation | High cardinality (> 50 levels) | Target encoding with smoothing or frequency encoding |
| **Datetime** | Timestamp gaps, seasonality, trend | Non-stationary drift, future leakage | Chronological split, lag/rolling features with `.shift(1)` |
